# Etapa 7 — Predictor de abandono

Análisis que respalda `src/ml_abandono.py` (decisión **D58** en `docs/decisiones.md`).

**Pregunta:** de los alumnos que vienen entrando al aula, ¿quiénes van a dejar de entrar en las próximas 2 semanas?
Sirve para que el tutor contacte a tiempo a quien está por dejar, en vez de enterarse cuando ya lleva semanas sin entrar.

**Objetivo (definición operativa).** A una fecha de corte *c*, un alumno de la base que tuvo **alguna interacción en los
últimos 14 días** *abandona* si **no tiene ninguna interacción en los 14 días siguientes** (c, c + 14].

**Variables:** solo la actividad (trazabilidad) hasta la fecha de corte. Las notas no se usan: son una foto al 24/10 y le
contarían al modelo cosas que todavía no habían pasado (D34).

El notebook solo **analiza**: todo el cálculo está en `src/ml_abandono.py`, que es lo que usa la app.
Para correrlo hace falta `pip install jupyter` (no está en `requirements.txt` porque la app no lo necesita).

In [1]:
import os
from pathlib import Path
if not Path("src").exists():          # el notebook está en notebooks/: subo a la raíz
    os.chdir("..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

from src import metricas as m
from src import ml_abandono as ab
from src.config import ABANDONO_HORIZONTE_DIAS as H, ABANDONO_SEMILLA

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
d = m.cargar()

## 1. Cómo se ve la cursada

Alumnos de la base que entraron cada semana. La cursada va del 05/08 al 24/10/2025 (12 semanas). La semana del 01/09 cae
en las tres materias (¿receso o semana sin clases?): es justamente donde más "abandonos" de 2 semanas aparecen.

In [2]:
ev = d.interacciones[d.interacciones["en_base"]]
semana = ev["fecha"] - pd.to_timedelta(ev["fecha"].dt.day_of_week, unit="D")
activos = ev.assign(semana=semana).groupby(["semana", "asignatura"])["id_alumno"].nunique().rename("alumnos").reset_index()
px.line(activos, x="semana", y="alumnos", color="asignatura", markers=True,
        title="Alumnos de la base que entraron cada semana")

## 2. Las fotos: de dónde salen los casos

No hay una fila "abandonó / no abandonó" en los datos: hay que construirla. Se toman **fotos** de la cursada cada 3 días
(no cada 7, para que caigan en distintos días de la semana: cada materia tiene su día fuerte) desde el día 14 hasta 14
días antes del último dato, para poder saber qué pasó después. En cada foto entran los alumnos que vienen entrando.

In [3]:
ds = ab.dataset(d)
por_foto = ds.groupby("fecha_corte").agg(alumnos=("abandona", "size"), abandonan=("abandona", "sum"),
                                         tasa=("abandona", "mean"))
por_foto["tasa"] = (100 * por_foto["tasa"]).round(1)
print(f"{len(ds)} casos, {ds['id_alumno'].nunique()} alumnos distintos, {len(por_foto)} fotos; "
      f"abandona el {100 * ds['abandona'].mean():.1f} %")
por_foto

9678 casos, 226 alumnos distintos, 18 fotos; abandona el 10.7 %


,alumnos,abandonan,tasa
fecha_corte,,,
2025-08-19,594,57,9.6
2025-08-22,582,77,13.2
2025-08-25,572,83,14.5
2025-08-28,573,94,16.4
2025-08-31,574,91,15.9
2025-09-03,552,69,12.5
2025-09-06,526,40,7.6
2025-09-09,495,37,7.5
2025-09-12,501,33,6.6


In [4]:
por_materia = ds.groupby("asignatura")["abandona"].agg(["size", "mean"]).rename(columns={"size": "casos", "mean": "tasa"})
por_materia["tasa"] = (100 * por_materia["tasa"]).round(1)
por_materia

,casos,tasa
asignatura,,
Algoritmos y Estructuras de Datos,2997,12.7
Probabilidad y Estadística,3327,10.3
Álgebra,3354,9.2


### ¿Los que "abandonan" vuelven?

"Dejar de entrar 14 días" no es la baja formal. Para las fotos que tienen al menos 4 semanas por delante, se mira si los
que abandonaron volvieron a entrar **después** de esas 2 semanas.

In [5]:
fin = ab.fin_datos(d)
vuelven = []
for corte, grupo in ds[ds["abandona"]].groupby("fecha_corte"):
    if corte + pd.Timedelta(days=2 * H) > fin:
        continue
    despues = ev[ev["fecha"] > corte + pd.Timedelta(days=H)]
    volvio = pd.MultiIndex.from_frame(grupo[ab.CLAVES]).isin(pd.MultiIndex.from_frame(despues[ab.CLAVES]))
    vuelven.append({"fecha_corte": corte, "abandonan": len(grupo), "vuelven_despues": int(volvio.sum())})
vuelven = pd.DataFrame(vuelven)
print(f"Volvieron a entrar más adelante: {vuelven['vuelven_despues'].sum()} de {vuelven['abandonan'].sum()} "
      f"({100 * vuelven['vuelven_despues'].sum() / vuelven['abandonan'].sum():.0f} %)")
vuelven

Volvieron a entrar más adelante: 435 de 745 (58 %)


,fecha_corte,abandonan,vuelven_despues
0,2025-08-19,57,36
1,2025-08-22,77,52
2,2025-08-25,83,64
3,2025-08-28,94,72
4,2025-08-31,91,68
5,2025-09-03,69,48
6,2025-09-06,40,22
7,2025-09-09,37,17
8,2025-09-12,33,10
9,2025-09-15,27,1


## 3. Variables: en qué se diferencian los que abandonan

Medianas de cada variable según el desenlace. Todas se calculan con datos hasta la fecha de la foto.

In [6]:
medianas = ds.groupby("abandona")[list(ab.VARIABLES)].median().T
medianas.columns = ["sigue", "abandona"]
medianas.insert(0, "descripcion", medianas.index.map(ab.VARIABLES))
medianas

,descripcion,sigue,abandona
dias_sin_actividad,Días desde la última vez que entró,2.000,5.0000
eventos_7d,Interacciones en los últimos 7 días,15.000,3.0000
eventos_14d,Interacciones en los últimos 14 días,37.000,17.0000
dias_activos_14d,Días con actividad en los últimos 14 días,4.000,2.0000
sesiones_14d,Sesiones en los últimos 14 días,5.000,2.0000
minutos_14d,Minutos en la plataforma en los últimos 14 días,29.400,10.3000
evaluaciones_14d,Interacciones de evaluación en los últimos 14 ...,3.000,1.0000
tendencia,Tendencia: últimas 2 semanas contra las 2 ante...,0.026,0.1485
pct_semanas_activas,% de las semanas de cursada en que entró,88.900,66.7000
eventos_total,Interacciones desde el inicio de la cursada,138.000,74.0000


In [7]:
px.histogram(ds, x="dias_sin_actividad", color="abandona", barmode="group", histnorm="percent", nbins=14,
             title="Días sin entrar a la fecha de la foto, según si después abandona")

In [8]:
tasa_dias = ds.groupby("dias_sin_actividad")["abandona"].agg(["size", "mean"]).rename(columns={"size": "casos", "mean": "tasa"})
tasa_dias["tasa"] = (100 * tasa_dias["tasa"]).round(1)
tasa_dias.T

dias_sin_actividad,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0
casos,2730.0,1711.0,1185.0,853.0,743.0,563.0,408.0,339.0,270.0,222.0,190.0,169.0,159.0,136.0
tasa,3.9,6.7,7.7,9.4,11.7,13.5,17.4,20.1,25.2,27.5,26.3,34.9,32.7,35.3


## 4. Comparación de modelos

`ab.entrenar(d)` hace todo:

1. Separa el **25 % de los alumnos** para prueba (todas sus fotos y materias): `GroupShuffleSplit` por `id_alumno`.
   Si un alumno estuviera en las dos partes, el modelo lo "reconocería" y la prueba sería optimista.
2. En el 75 % restante, **validación cruzada por alumno** (`GroupKFold`, 5 partes): con esas predicciones elige el
   umbral de menor costo para cada modelo y el modelo de menor costo.
3. Mide los tres en prueba (regresión logística, Random Forest y la regla "más de 7 días sin entrar").
4. Reentrena el elegido con todos los datos.

In [9]:
mod = ab.entrenar(d)
print(f"Elegido: {mod.nombre}, umbral {mod.umbral}")
mod.comparacion.drop(columns=["elegido"])

Elegido: Regresión logística, umbral 0.14


,modelo,conjunto,filas,abandonos,auc_roc,auc_pr,brier,precision,recall,pct_marcados,costo,costo_por_100
0,Regresión logística,validación cruzada,7367,766,0.776,0.306,0.0826,26.5,59.3,23.2,2816.0,38.2
1,Regresión logística,prueba,2311,266,0.796,0.340,0.0885,30.0,64.3,24.7,874.0,37.8
2,Random Forest,validación cruzada,7367,766,0.785,0.311,0.0823,23.9,68.7,29.9,2876.0,39.0
3,Random Forest,prueba,2311,266,0.810,0.327,0.0881,27.4,75.2,31.6,861.0,37.3
4,Regla simple,validación cruzada,7367,766,0.712,0.215,NaN,28.0,32.1,11.9,3233.0,43.9
5,Regla simple,prueba,2311,266,0.714,0.278,NaN,34.5,34.6,11.6,1045.0,45.2


## 5. Validación temporal

La validación por alumno mezcla semanas: el modelo ve fotos de octubre al entrenar. En la práctica se va a usar para
predecir **semanas que todavía no pasaron**, así que también se prueba entrenando con las fotos hasta el 20/09 y midiendo
en las posteriores (con alumnos de prueba distintos, para no mezclar las dos cosas).

In [10]:
X, y = ds[list(ab.VARIABLES)].astype(float), ds["abandona"].to_numpy()
e, p = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=ABANDONO_SEMILLA).split(X, y, ds["id_alumno"]))
temprano = ds["fecha_corte"] <= "2025-09-20"
ent = np.zeros(len(ds), bool); ent[e] = True
filas = []
for clave, est in ab.modelos().items():
    ajuste = clone(est).fit(X[ent & temprano], y[ent & temprano])
    for nombre, mascara in [("misma época (alumnos de prueba, hasta 20/09)", ~ent & temprano),
                            ("después (alumnos de prueba, desde 21/09)", ~ent & ~temprano)]:
        prob = ajuste.predict_proba(X[mascara])[:, 1]
        filas.append({"modelo": ab.NOMBRES_MODELOS[clave], "prueba": nombre,
                      "auc_roc": roc_auc_score(y[mascara], prob), "auc_pr": average_precision_score(y[mascara], prob)})
for nombre, mascara in [("misma época (alumnos de prueba, hasta 20/09)", ~ent & temprano),
                        ("después (alumnos de prueba, desde 21/09)", ~ent & ~temprano)]:
    dias = X.loc[mascara, "dias_sin_actividad"]
    filas.append({"modelo": ab.REGLA, "prueba": nombre, "auc_roc": roc_auc_score(y[mascara], dias),
                  "auc_pr": average_precision_score(y[mascara], dias)})
pd.DataFrame(filas).round(3).pivot(index="modelo", columns="prueba", values=["auc_roc", "auc_pr"])

auc_roc                                                                                auc_pr                                             
prueba              después (alumnos de prueba, desde 21/09) misma época (alumnos de prueba, hasta 20/09) después (alumnos de prueba, desde 21/09) misma época (alumnos de prueba, hasta 20/09)
modelo                                                                                                                                                                                         
Random Forest                                          0.826                                        0.812                                    0.329                                        0.357
Regla simple                                           0.756                                        0.697                                    0.283                                        0.281
Regresión logística                                    0.801                                        0.807                                    0.313                                        0.363

## 6. Calibración: ¿la probabilidad se puede leer como probabilidad?

En los alumnos de prueba, se agrupan los casos en 10 partes según la probabilidad predicha y se compara con el % que
abandonó de verdad. Si están cerca, "30 %" quiere decir "de cada 10 así, abandonan 3".

In [11]:
ajuste = clone(ab.modelos()[mod.clave]).fit(X.iloc[e], y[e])
prob = ajuste.predict_proba(X.iloc[p])[:, 1]
cal = pd.DataFrame({"prob": prob, "real": y[p]})
cal["decil"] = pd.qcut(cal["prob"].rank(method="first"), 10, labels=False) + 1
cal = cal.groupby("decil").agg(predicha=("prob", "mean"), real=("real", "mean"), casos=("real", "size"))
cal[["predicha", "real"]] = (100 * cal[["predicha", "real"]]).round(1)
fig = px.scatter(cal, x="predicha", y="real", size="casos", title="Calibración en los alumnos de prueba (%)")
fig.add_shape(type="line", x0=0, y0=0, x1=60, y1=60, line=dict(dash="dash"))
fig.show()
cal

,predicha,real,casos
decil,,,
1,1.5,1.7,232
2,2.6,1.7,231
3,3.6,1.3,231
4,4.5,4.8,231
5,5.9,4.8,231
6,7.8,10.8,231
7,10.2,8.2,231
8,13.8,16.5,231
9,20.3,29.9,231


## 7. El umbral según el costo

El umbral sale de cuánto cuesta cada error (como en la clase 10). Con costo 5 a 1 (no detectar a quien abandona cuesta
5 veces más que contactar de más) el umbral es bajo. La tabla muestra qué pasaría con otras proporciones: es la
discusión que hay que tener con la coordinación, no una decisión técnica.

In [12]:
oof_y, oof_p = y[p], prob
filas = []
for fn in [1, 2, 3, 5, 10]:
    u = ab.elegir_umbral(oof_y, oof_p, costo_fn=fn, costo_fp=1)
    t = ab.tabla_umbral(oof_y, oof_p, costo_fn=fn).set_index("umbral").loc[u]
    filas.append({"costo_fn : costo_fp": f"{fn} : 1", "umbral": u, "% marcados": t["pct_marcados"],
                  "precisión": t["precision"], "recall": t["recall"]})
pd.DataFrame(filas)

,costo_fn : costo_fp,umbral,% marcados,precisión,recall
0,1 : 1,0.45,1.51,60.00,7.89
1,2 : 1,0.37,3.33,53.25,15.41
2,3 : 1,0.17,19.17,32.96,54.89
3,5 : 1,0.13,26.48,29.41,67.67
4,10 : 1,0.08,43.75,21.96,83.46


In [13]:
curva = mod.curva.assign(umbral=lambda c: 100 * c["umbral"])
fig = px.line(curva, x="umbral", y="costo", title="Costo total según el umbral (alumnos de prueba, costo 5 : 1)")
fig.add_vline(x=100 * mod.umbral, line_dash="dash")
fig

## 8. Qué variables usa el modelo

Importancia por permutación en los alumnos de prueba: cuánto empeora el AUC PR si se mezclan al azar los valores de
una variable. Las variables están muy correlacionadas entre sí (interacciones de 7 y de 14 días, sesiones, minutos), así
que cuando una "no importa" suele ser porque otra ya dice lo mismo.

In [14]:
mod.importancia

,variable,descripcion,importancia,desvio,coeficiente
0,dias_activos_14d,Días con actividad en los últimos 14 días,0.0379,0.0072,0.003
1,eventos_total,Interacciones desde el inicio de la cursada,0.0372,0.0094,-0.172
2,pct_semanas_activas,% de las semanas de cursada en que entró,0.0179,0.0054,0.159
3,dias_sin_actividad,Días desde la última vez que entró,0.0127,0.0054,-0.207
4,eventos_7d,Interacciones en los últimos 7 días,0.0121,0.0043,-0.020
5,sesiones_14d,Sesiones en los últimos 14 días,0.0069,0.0033,0.008
6,tendencia,Tendencia: últimas 2 semanas contra las 2 ante...,0.0043,0.0038,-0.405
7,evaluaciones_14d,Interacciones de evaluación en los últimos 14 ...,-0.0006,0.0009,0.163
8,minutos_14d,Minutos en la plataforma en los últimos 14 días,-0.0009,0.0009,-0.373
9,eventos_14d,Interacciones en los últimos 14 días,-0.0017,0.0013,-0.120


## 9. A la fecha de corte por defecto (24/10/2025)

In [15]:
ab.resumen_riesgo(d, modelo=mod)

,asignatura,base,cursando,riesgo_alto,riesgo_medio,riesgo_bajo,ya_abandonaron,sin_actividad,pct_riesgo_alto,regla_simple,probabilidad_promedio
0,Algoritmos y Estructuras de Datos,204,131,41,33,57,71,2,31.3,32,11.7
1,Probabilidad y Estadística,216,150,32,32,86,63,3,21.3,30,9.1
2,Álgebra,217,169,55,40,74,46,2,32.5,56,13.0


In [16]:
ab.resumen_riesgo(d, "2025-09-15", modelo=mod)

,asignatura,base,cursando,riesgo_alto,riesgo_medio,riesgo_bajo,ya_abandonaron,sin_actividad,pct_riesgo_alto,regla_simple,probabilidad_promedio,abandonaron_despues,aciertos_alto
0,Algoritmos y Estructuras de Datos,204,156,39,41,76,45,3,25.0,15,10.3,12,10
1,Probabilidad y Estadística,216,166,44,33,89,44,6,26.5,11,9.7,7,5
2,Álgebra,217,178,32,49,97,35,4,18.0,7,9.5,8,5


## Conclusiones

* **Casos:** 18 fotos (19/08 a 09/10) dan 9.678 casos de 226 alumnos distintos; abandona el **10,7 %** (de 5 % a
  16 % según la semana; el pico es alrededor de la semana del 01/09, que tiene poca actividad en las tres materias).
* **Abandonar 14 días no es irse:** el **58 %** de los que dejaron de entrar 2 semanas volvió más adelante (casi todos
  los de fines de agosto). Para octubre, cuando ya no queda tanto tiempo para volver, el abandono se parece más a uno
  real. Por eso el resultado se usa para **priorizar contactos**, no para dar a nadie por perdido.
* **Los modelos mejoran a la regla simple, pero no por mucho.** En los alumnos de prueba: AUC ROC 0,80 (logística) y
  0,81 (Random Forest) contra 0,71 de "más de 7 días sin entrar"; con el umbral por costo detectan al 64-75 % de los
  que abandonan (la regla, al 35 %) a cambio de marcar a 1 de cada 4 alumnos. El costo baja de 45 a 37-38 cada 100 casos.
* **Se elige la regresión logística:** menor costo en la validación cruzada (en prueba el Random Forest queda apenas
  mejor, dentro del ruido), es más simple de explicar y sus probabilidades están bien calibradas (los deciles caen
  cerca de la diagonal), así que "30 %" se puede leer como "3 de cada 10".
* **Se sostiene en el tiempo:** entrenando hasta el 20/09 y midiendo después, el AUC queda igual (0,80-0,83).
* **Qué mira:** cuántos días entró en las últimas 2 semanas, cuánto entró desde el inicio y en qué % de las semanas,
  y hace cuántos días no entra. Los días sin entrar solos ya separan bastante (con 0 días abandona el 4 %; con 13, el 35 %):
  el modelo gana sumando la regularidad.
* **Umbral:** con costo 5 : 1 el umbral es 13-14 %. Es una decisión de la coordinación: con 3 : 1 casi no cambia, y
  con 2 : 1 el umbral sube a 37 % y se marca solo al 3 % (pero se detecta al 15 %).
* **Limitaciones:** una sola cursada y 226 personas; los casos de un mismo alumno no son independientes (por eso la
  validación es por alumno); el objetivo es "no entrar al aula", no la baja formal; sin notas, a propósito, para no usar
  información del futuro.